# ShadowTag v2 - Vertex AI Workbench

Dual-layer steganographic watermarking for content authenticity.

**Version**: 2.0.0  
**Mode**: Bourne/160 STRICT (PNKLN)

## Overview

- **Video Layer**: DCT mid-band pixel watermarking (H.264/H.265 resilient)
- **Audio Layer**: Ultrasonic spread-spectrum (≥19.5 kHz)
- **Blockchain**: Tamper-evident receipts (Polygon/Ethereum)

## Setup

In [ ]:
%%bash
# Environment bootstrap
set -euo pipefail

# Install system dependencies
sudo apt-get update -qq
sudo apt-get install -y -qq libsndfile1 ffmpeg libgl1-mesa-glx libglib2.0-0

# Install Python packages
pip install -q opencv-python numpy pydub soundfile librosa scipy typer rich \
    web3 eth-account google-cloud-secret-manager cryptography

echo "✅ Environment ready"

In [ ]:
# Configuration
PROJECT_ID = "YOUR_GCP_PROJECT_ID"  # EDIT THIS
REGION = "us-central1"
BLOCKCHAIN_ENABLED = False  # Set to True to enable blockchain receipts

print(f"Project: {PROJECT_ID}")
print(f"Region: {REGION}")
print(f"Blockchain: {'enabled' if BLOCKCHAIN_ENABLED else 'disabled'}")

## Video Watermarking (Pixel Layer)

In [ ]:
# Import ShadowTag v2 modules
import sys

sys.path.insert(0, "..")  # Add parent directory to path

from pathlib import Path

from shadowtag_v2.video_stego import (
    VideoWatermarkConfig,
    embed_video_watermark,
    extract_video_watermark,
)

print("✅ Video stego module loaded")

In [ ]:
# Example: Embed video watermark
INPUT_VIDEO = Path("sample_video.mp4")  # Provide your video
OUTPUT_VIDEO = Path("watermarked_video.mp4")
PROMPT = "A sunset over mountains with vibrant orange and purple hues"

if INPUT_VIDEO.exists():
    config = VideoWatermarkConfig()
    result = embed_video_watermark(INPUT_VIDEO, OUTPUT_VIDEO, PROMPT, config)

    print("✅ Video watermark embedded!")
    print(f"Frames processed: {result['frames']}")
    print(f"Prompt hash: {result['prompt_hash']}")
else:
    print("⚠️  Upload sample_video.mp4 first")

In [ ]:
# Example: Verify video watermark
if OUTPUT_VIDEO.exists():
    verify_result = extract_video_watermark(OUTPUT_VIDEO, PROMPT, VideoWatermarkConfig())

    if verify_result.get("verified"):
        print("✅ VERIFIED")
        print(f"BER: {verify_result.get('bit_error_rate', 0):.4f}")
    else:
        print("❌ NOT VERIFIED")
        print(f"Extracted: {verify_result['extracted_hash']}")
        print(f"Expected: {verify_result.get('expected_hash')}")

## Audio Watermarking (Ultrasonic Layer)

In [ ]:
from shadowtag_v2.audio_stego import (
    AudioWatermarkConfig,
    embed_audio_watermark,
    extract_audio_watermark,
)

print("✅ Audio stego module loaded")

In [ ]:
# Example: Embed audio watermark
INPUT_AUDIO = Path("sample_audio.wav")  # 48 kHz stereo WAV
OUTPUT_AUDIO = Path("watermarked_audio.wav")

if INPUT_AUDIO.exists():
    config = AudioWatermarkConfig()
    result = embed_audio_watermark(INPUT_AUDIO, OUTPUT_AUDIO, PROMPT, config)

    print("✅ Audio watermark embedded!")
    print(f"Sample rate: {result['sample_rate']} Hz")
    print(f"Carrier freq: {result['carrier_freq']} Hz")
else:
    print("⚠️  Upload sample_audio.wav first (48 kHz)")

In [ ]:
# Example: Verify audio watermark
if OUTPUT_AUDIO.exists():
    verify_result = extract_audio_watermark(OUTPUT_AUDIO, PROMPT, AudioWatermarkConfig())

    if verify_result.get("verified"):
        print("✅ VERIFIED")
    else:
        print("⚠️  Verification uncertain (audio watermarks less robust)")
        print(f"Extracted: {verify_result['extracted_hash']}")

## Blockchain Receipts

In [ ]:
from shadowtag_v2.receipt_chain import BlockchainConfig, ChainType, create_blockchain_receipt

print("✅ Blockchain receipt module loaded")

In [ ]:
# Example: Create blockchain receipt (requires configuration)
if BLOCKCHAIN_ENABLED:
    config = BlockchainConfig(
        chain=ChainType.POLYGON_MUMBAI,  # Testnet
        gcp_secret_name="blockchain-private-key",
        gcp_project_id=PROJECT_ID,
    )

    receipt = create_blockchain_receipt(
        prompt=PROMPT, config=config, metadata={"type": "demo", "version": "2.0.0"}
    )

    print("✅ Blockchain receipt created!")
    print(f"TX Hash: {receipt['tx_hash']}")
    print(f"Chain: {receipt['chain']}")
else:
    print("⚠️  Blockchain disabled. Set BLOCKCHAIN_ENABLED=True and configure GCP secrets.")

## FastAPI Service Integration

In [ ]:
%%bash
# Start FastAPI server in background
cd ..
uvicorn api.main:app --host 0.0.0.0 --port 8080 &
sleep 3
echo "✅ API server started on port 8080"

In [ ]:
# Test API health endpoint
import requests

response = requests.get("http://localhost:8080/health")
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

## PNKLN Operational Checklist

### Security Gates
- ✅ No plaintext secrets (use GCP Secret Manager)
- ✅ TLS enforced for all network operations
- ✅ Input validation on all user-provided data
- ✅ Encryption at rest and in transit

### Quality Gates
- ✅ Test coverage ≥ 98%
- ✅ Static analysis passing (ruff, mypy)
- ✅ Two green CI runs before merge
- ✅ Pre-commit hooks enabled

### ROI Gate
- Target: ≥3× return in 18 months
- Kill-switch if ROI threshold violated

### Legal/Export Compliance
- Dual-use technology exported as 'content authenticity'
- LEO/GOV/ITAR: "LEO/GOV/ITAR in effect" (details gated)

### Moat
- Data flywheel (watermark verification corpus)
- Process power (patent-pending dual-layer technique)
- Network effects (creator ecosystem)